In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

# ── 1. CHARGEMENT DES DONNÉES ──────────────────────────────────
# df = pd.read_csv("Airplane_Crashes_and_Fatalities_upto_2023.csv")

# Données simulées (même structure que le vrai dataset)
np.random.seed(42)
n = 5000
aboard     = np.random.lognormal(3.5, 1.2, n).astype(int).clip(1, 500)
fatalities = np.minimum((aboard * np.random.beta(2, 3, n)).astype(int), aboard)

df = pd.DataFrame({
    'Year'      : np.random.randint(1908, 2024, n),
    'Operator'  : np.random.choice(['American Airlines','Air France','Aeroflot','Military','Private'], n),
    'Region'    : np.random.choice(['North America','Europe','Asia','South America','Africa'], n),
    'Aboard'    : aboard,
    'Fatalities': fatalities,
})

# ── 2. NETTOYAGE ───────────────────────────────────────────────
df['Fatalities'] = pd.to_numeric(df['Fatalities'], errors='coerce').fillna(0).astype(int)
df['Aboard']     = pd.to_numeric(df['Aboard'],     errors='coerce').fillna(0).astype(int)
df['Survivors']  = df['Aboard'] - df['Fatalities']
df['Survival_Rate'] = df['Survivors'] / df['Aboard']
df['Decade']     = (df['Year'] // 10 * 10).astype(str) + 's'
df = df[df['Aboard'] > 0]

# ── 3. STATISTIQUES DE BASE ────────────────────────────────────
print(f"Total crashs      : {len(df):,}")
print(f"Total fatalités   : {df['Fatalities'].sum():,}")
print(f"Taux de survie    : {df['Survival_Rate'].mean()*100:.1f}%")
print(f"Moyenne fatalités : {df['Fatalities'].mean():.1f}")
print(f"Médiane fatalités : {df['Fatalities'].median():.1f}")
print(f"Écart-type        : {df['Fatalities'].std():.1f}")

# ── 4. TEST STATISTIQUE ────────────────────────────────────────
pre  = df[df['Year'] <  1970]['Fatalities']
post = df[df['Year'] >= 1970]['Fatalities']
t, p = stats.ttest_ind(pre, post, equal_var=False)
print(f"\nT-test avant/après 1970 : t={t:.2f}, p={p:.4f}")
print("→", "Différence significative" if p < 0.05 else "Pas de différence significative")

# ── 5. VISUALISATIONS ─────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle("Analyse des Crashs Aériens", fontsize=15, fontweight='bold')

# Crashs par année
crashes_year = df.groupby('Year').size()
axes[0,0].plot(crashes_year.index, crashes_year.values, color='steelblue')
axes[0,0].set_title('Crashs par Année')
axes[0,0].set_xlabel('Année'); axes[0,0].set_ylabel('Nombre')
axes[0,0].grid(True)

# Distribution des fatalités
axes[0,1].hist(df['Fatalities'].clip(upper=200), bins=40, color='tomato', edgecolor='white')
axes[0,1].axvline(df['Fatalities'].mean(),   color='blue',  linestyle='--', label='Moyenne')
axes[0,1].axvline(df['Fatalities'].median(), color='green', linestyle='--', label='Médiane')
axes[0,1].set_title('Distribution des Fatalités')
axes[0,1].set_xlabel('Fatalités'); axes[0,1].legend(); axes[0,1].grid(True)

# Crashs par région
region_counts = df['Region'].value_counts()
axes[1,0].barh(region_counts.index, region_counts.values, color='mediumpurple')
axes[1,0].set_title('Crashs par Région')
axes[1,0].set_xlabel('Nombre de crashs'); axes[1,0].grid(True, axis='x')

# Taux de survie par décennie
surv_decade = df.groupby('Decade')['Survival_Rate'].mean().sort_index() * 100
axes[1,1].bar(surv_decade.index, surv_decade.values, color='mediumseagreen', edgecolor='white')
axes[1,1].set_title('Taux de Survie par Décennie (%)')
axes[1,1].set_xlabel('Décennie'); axes[1,1].set_ylabel('%')
axes[1,1].tick_params(axis='x', rotation=45); axes[1,1].grid(True, axis='y')

plt.tight_layout()
plt.savefig('airplane_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
